Work is [in progress](https://www.wikidata.org/wiki/Wikidata:WikiProject_Indigenous_peoples_of_North_America/Data_model) to develop a series of data models organizing information about indigenous peoples, governments, and lands. In addition to information content about the models designed for humans, we need a technical encoding of data models that can be used to evaluate conformance of items to the models.

This majority of this work is in creating the schemas themselves. The notebook works through a part of that process using the Shape Expression method that Wikidata uses behind their still developing Entity Schemas ("E" identifiers). It is a potential new class that I may build into my wbmaker Python package at some point. It takes a QID for an item we want to test and an EID for a schema we want to test the item against. It uses PyShEx to evaluate these and yield a result object we can print out or do something else with.

In future, I might tweak this to do some other things like take a SPARQL query input to return and evaluate multiple items and report out on what conforms vs. doesn't conform to the schema.

In [1]:
from pyshex import ShExEvaluator
import requests
import os

class WikidataShExValidator:
    def __init__(self, qid: str = None, eid: str = None, user_agent: str = None, 
                 schema_text: str = None, schema_file: str = None,
                 rdf_text: str = None, rdf_file: str = None):
        """
        Initialize the validator with an optional user agent.
        
        Args:
            qid: Wikidata item ID (e.g., 'Q123'). Optional if rdf_text or rdf_file provided.
            eid: EntitySchema ID (e.g., 'E502'). Optional if schema_text or schema_file provided.
            user_agent: Custom user agent string. If None, uses default.
            schema_text: ShExC schema as a string (alternative to eid)
            schema_file: Path to local file containing ShExC schema (alternative to eid)
            rdf_text: RDF data as a string (alternative to qid)
            rdf_file: Path to local file containing RDF data (alternative to qid)
        """
        if user_agent is None:
            user_agent = 'WikidataShExValidator/1.0'
        
        self.headers = {
            "User-Agent": user_agent
        }

        self.qid = qid
        self.eid = eid
        self.schema_text = schema_text
        self.schema_file = schema_file
        self.rdf_text = rdf_text
        self.rdf_file = rdf_file
        self.shexc = None
        self.rdf = None
        self.results = None
    
    def fetch_entityschema(self):
        """
        Fetch ShExC text for a Wikidata EntitySchema (e.g., 'E502')
        using Special:EntitySchemaText, or load from local sources.
        """
        if self.schema_text:
            self.shexc = self.schema_text
        elif self.schema_file:
            with open(self.schema_file, 'r', encoding='utf-8') as f:
                self.shexc = f.read()
        elif self.eid:
            url = f"https://www.wikidata.org/wiki/Special:EntitySchemaText/{self.eid}"
            resp = requests.get(url, headers=self.headers)
            resp.raise_for_status()
            self.shexc = resp.text
        else:
            raise ValueError("No schema source provided. Specify eid, schema_text, or schema_file.")
        return self

    def fetch_rdf(self):
        """
        Fetch RDF data for the Wikidata item in Turtle format, or load from local sources.
        """
        if self.rdf_text:
            self.rdf = self.rdf_text
        elif self.rdf_file:
            with open(self.rdf_file, 'r', encoding='utf-8') as f:
                self.rdf = f.read()
        elif self.qid:
            rdf_url = f"https://www.wikidata.org/wiki/Special:EntityData/{self.qid}.ttl"
            resp = requests.get(rdf_url, headers=self.headers)
            resp.raise_for_status()
            self.rdf = resp.text
        else:
            raise ValueError("No RDF source provided. Specify qid, rdf_text, or rdf_file.")
        return self
    
    def eval_item(self):
        """
        Evaluate a Wikidata item against an EntitySchema.
        Requires that fetch_entityschema() and fetch_rdf() have been called first.
        """
        if self.shexc is None:
            raise ValueError("Schema not fetched. Call fetch_entityschema() first.")
        if self.rdf is None:
            raise ValueError("RDF not fetched. Call fetch_rdf() first.")
        
        # Determine focus node
        if self.qid:
            focus = f"http://www.wikidata.org/entity/{self.qid}"
        else:
            focus = None  # PyShEx will attempt to infer
            
        self.results = ShExEvaluator(
            rdf=self.rdf,
            schema=self.shexc,
            focus=focus
        ).evaluate()
        return self
    
    def validate(self):
        """
        Convenience method: fetch schema, fetch RDF, and evaluate in one call.
        Returns self to allow chaining or access to results.
        """
        self.fetch_entityschema()
        self.fetch_rdf()
        self.eval_item()
        return self

In [18]:
test_cases = {
    'Cherokee Nation': 'Q14708404',
    'Something Else': 'Q736809',
    'Chilkat Indian Village': 'Q137738323'
}

print("Testing all cases with current schema:")
print("=" * 70)

for name, qid in test_cases.items():
    print(f"\n{name} ({qid}):")
    v = WikidataShExValidator(
        qid=qid,
        schema_file="FederallyRecognizedTribe.shex",
        user_agent=os.environ['WB_BOT_USER_AGENT']
    ).validate()
    
    for r in v.results:
        print(f"  Conforms: {r.result}")
        if not r.result:
            print(f"  Reason (first 300 chars): {str(r.reason)[:300]}")

Testing all cases with current schema:

Cherokee Nation (Q14708404):
  Conforms: True

Something Else (Q736809):
  Conforms: False
  Reason (first 300 chars):   Testing wd:Q736809 against shape FederallyRecognizedTribe
       No matching triples found for predicate wdt:P30

Chilkat Indian Village (Q137738323):
  Conforms: True
